# Recurrent Neural Networks (RNN) — Fundamentals with PyTorch

This notebook covers the core theory and building blocks of Recurrent Neural Networks using PyTorch:

1. **Why RNNs?** — Limitations of feedforward networks for sequential data
2. **Vanilla RNN** — Architecture, forward pass, and hidden state dynamics
3. **Vanishing / Exploding Gradients** — The fundamental training challenge
4. **LSTM (Long Short-Term Memory)** — Gates and cell state
5. **GRU (Gated Recurrent Unit)** — Simplified gating mechanism
6. **Bidirectional RNNs** — Capturing context from both directions
7. **Hands-on**: Building and running each variant in PyTorch

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

---
## 1. Why RNNs?

Feedforward networks treat each input independently — they have **no memory** of previous inputs. For sequential data (text, time series, audio), the **order and context** matter.

RNNs solve this by maintaining a **hidden state** $h_t$ that is updated at every timestep:

$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

where:
- $x_t$ is the input at timestep $t$
- $h_{t-1}$ is the previous hidden state
- $W_{ih}$, $W_{hh}$ are learnable weight matrices

---
## 2. Vanilla RNN in PyTorch

Let's start by understanding PyTorch's `nn.RNN` module.

In [ ]:
# Parameters
input_size  = 10   # Feature dimension at each timestep
hidden_size = 20   # Hidden state dimension
num_layers  = 1    # Number of stacked RNN layers
seq_len     = 5    # Sequence length
batch_size  = 3

rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size,
             num_layers=num_layers, batch_first=True)

# Random input: (batch, seq_len, input_size)
x = torch.randn(batch_size, seq_len, input_size)

# Initial hidden state: (num_layers, batch, hidden_size)
h0 = torch.zeros(num_layers, batch_size, hidden_size)

output, h_n = rnn(x, h0)

print(f"Input shape     : {x.shape}")
print(f"Output shape    : {output.shape}")   # (batch, seq_len, hidden_size)
print(f"Hidden(n) shape : {h_n.shape}")      # (num_layers, batch, hidden_size)
print(f"\noutput[:, -1, :] == h_n[-1]: {torch.allclose(output[:, -1, :], h_n[-1])}")

**Key Observations:**
- `output` contains hidden states at **every timestep** — shape `(B, T, H)`
- `h_n` is the hidden state at the **last timestep** — shape `(num_layers, B, H)`
- For a single-layer RNN, `output[:, -1, :]` equals `h_n[-1]`

In [ ]:
# Manual RNN forward pass to understand the mechanics
class ManualRNNCell:
    """Step through an RNN manually to visualize hidden state evolution."""

    def __init__(self, input_size, hidden_size):
        self.W_ih = torch.randn(hidden_size, input_size) * 0.1
        self.W_hh = torch.randn(hidden_size, hidden_size) * 0.1
        self.b = torch.zeros(hidden_size)

    def forward(self, x_seq):
        """x_seq: (seq_len, input_size)"""
        seq_len = x_seq.shape[0]
        h = torch.zeros(self.W_hh.shape[0])
        hidden_states = [h.clone()]

        for t in range(seq_len):
            h = torch.tanh(self.W_ih @ x_seq[t] + self.W_hh @ h + self.b)
            hidden_states.append(h.clone())

        return torch.stack(hidden_states)  # (seq_len+1, hidden_size)


cell = ManualRNNCell(input_size=4, hidden_size=8)
x_seq = torch.randn(10, 4)  # 10 timesteps, 4 features
states = cell.forward(x_seq)

plt.figure(figsize=(12, 4))
plt.imshow(states.detach().numpy().T, aspect='auto', cmap='coolwarm')
plt.colorbar(label='Activation')
plt.xlabel('Timestep')
plt.ylabel('Hidden Unit')
plt.title('Hidden State Evolution Over Time (Manual RNN)')
plt.tight_layout()
plt.show()

---
## 3. The Vanishing / Exploding Gradient Problem

During backpropagation through time (BPTT), gradients are multiplied by $W_{hh}$ at every step:

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

- If $\|W_{hh}\| < 1$ (spectral norm), gradients **vanish** exponentially
- If $\|W_{hh}\| > 1$, gradients **explode**

This makes it hard for vanilla RNNs to learn **long-range dependencies**.

In [ ]:
# Demonstrating gradient magnitude decay over timesteps
def simulate_gradient_flow(weight_scale, timesteps=50):
    W = torch.randn(32, 32) * weight_scale
    gradient = torch.ones(32)
    magnitudes = [gradient.norm().item()]
    for _ in range(timesteps):
        gradient = W.T @ gradient
        magnitudes.append(gradient.norm().item())
    return magnitudes

plt.figure(figsize=(12, 5))
for scale, label in [(0.1, 'Small W (vanishing)'),
                      (0.5, 'Medium W'),
                      (1.0, 'Large W (exploding)')]:
    mags = simulate_gradient_flow(scale)
    plt.plot(mags, label=label, linewidth=2)

plt.yscale('log')
plt.xlabel('Timesteps Back')
plt.ylabel('Gradient Norm (log scale)')
plt.title('Gradient Flow Through Time — Vanishing vs Exploding')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. LSTM (Long Short-Term Memory)

LSTMs address vanishing gradients with a **cell state** $C_t$ and three gates:

| Gate | Formula | Purpose |
|------|---------|---------|
| **Forget** | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | What to discard from cell state |
| **Input** | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | What new info to store |
| **Output** | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | What to output from cell state |

Cell state update:
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
$$h_t = o_t \odot \tanh(C_t)$$

The cell state provides a **highway** for gradients to flow unchanged.

In [ ]:
lstm = nn.LSTM(input_size=10, hidden_size=20,
               num_layers=2, batch_first=True)

x = torch.randn(3, 5, 10)  # (batch=3, seq_len=5, features=10)
h0 = torch.zeros(2, 3, 20)  # (num_layers, batch, hidden)
c0 = torch.zeros(2, 3, 20)  # cell state

output, (h_n, c_n) = lstm(x, (h0, c0))

print(f"Output shape     : {output.shape}")   # (3, 5, 20)
print(f"Hidden(n) shape  : {h_n.shape}")      # (2, 3, 20)
print(f"Cell(n) shape    : {c_n.shape}")       # (2, 3, 20)
print(f"\nLSTM has {sum(p.numel() for p in lstm.parameters()):,} parameters")
print(f"(~4x a vanilla RNN with same dimensions due to 4 gate matrices)")

In [ ]:
# Visualize LSTM gate activations
lstm_cell = nn.LSTMCell(input_size=4, hidden_size=8)

seq = torch.randn(20, 4)  # 20 timesteps
h, c = torch.zeros(1, 8), torch.zeros(1, 8)

hidden_history = []
cell_history = []

for t in range(20):
    h, c = lstm_cell(seq[t:t+1], (h, c))
    hidden_history.append(h.detach().squeeze().numpy())
    cell_history.append(c.detach().squeeze().numpy())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(np.array(hidden_history).T, aspect='auto', cmap='viridis')
axes[0].set_title('LSTM Hidden State (h) Over Time')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Hidden Unit')

axes[1].imshow(np.array(cell_history).T, aspect='auto', cmap='magma')
axes[1].set_title('LSTM Cell State (C) Over Time')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Cell Unit')

plt.tight_layout()
plt.show()

---
## 5. GRU (Gated Recurrent Unit)

GRU simplifies LSTM by merging the cell and hidden state, using only two gates:

| Gate | Formula | Purpose |
|------|---------|---------|
| **Reset** | $r_t = \sigma(W_r [h_{t-1}, x_t])$ | How much past to forget |
| **Update** | $z_t = \sigma(W_z [h_{t-1}, x_t])$ | Balance of old vs new |

$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t])$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

**Fewer parameters** than LSTM (~75%), often comparable performance.

In [ ]:
gru = nn.GRU(input_size=10, hidden_size=20,
             num_layers=2, batch_first=True)

x = torch.randn(3, 5, 10)
h0 = torch.zeros(2, 3, 20)

output, h_n = gru(x, h0)

print(f"Output shape    : {output.shape}")
print(f"Hidden(n) shape : {h_n.shape}")
print(f"\nGRU has {sum(p.numel() for p in gru.parameters()):,} parameters")

# Compare parameter counts
rnn_params  = sum(p.numel() for p in nn.RNN(10, 20, 2).parameters())
lstm_params = sum(p.numel() for p in nn.LSTM(10, 20, 2).parameters())
gru_params  = sum(p.numel() for p in nn.GRU(10, 20, 2).parameters())

print(f"\nParameter comparison (input=10, hidden=20, layers=2):")
print(f"  Vanilla RNN : {rnn_params:,}")
print(f"  LSTM        : {lstm_params:,}  ({lstm_params/rnn_params:.1f}x RNN)")
print(f"  GRU         : {gru_params:,}  ({gru_params/rnn_params:.1f}x RNN)")

---
## 6. Bidirectional RNNs

A bidirectional RNN processes the sequence in **both directions** and concatenates the outputs:

- Forward:  $\overrightarrow{h_t}$ processes $x_1 \to x_T$
- Backward: $\overleftarrow{h_t}$ processes $x_T \to x_1$
- Output:   $h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$ — dimension doubles

Bidirectional RNNs capture **both past and future context** at each timestep.

In [ ]:
bi_lstm = nn.LSTM(input_size=10, hidden_size=20,
                  num_layers=2, batch_first=True, bidirectional=True)

x = torch.randn(3, 5, 10)
output, (h_n, c_n) = bi_lstm(x)

print(f"Output shape    : {output.shape}")   # (3, 5, 40)  — hidden * 2
print(f"Hidden(n) shape : {h_n.shape}")      # (4, 3, 20)  — num_layers * 2

# Extracting forward and backward final hidden states from the last layer
h_forward  = h_n[-2]   # Forward direction, last layer
h_backward = h_n[-1]   # Backward direction, last layer
h_combined = torch.cat([h_forward, h_backward], dim=1)  # (batch, hidden*2)

print(f"\nCombined hidden : {h_combined.shape}")
print(f"Bi-LSTM params  : {sum(p.numel() for p in bi_lstm.parameters()):,}")

---
## 7. Practical Example: Sequence Classification

Let's build a simple sequence classifier that learns to distinguish **increasing vs decreasing** number sequences.

In [ ]:
# Generate synthetic data: increasing (label=1) vs decreasing (label=0) sequences
def generate_sequences(n_samples=1000, seq_len=10):
    X, y = [], []
    for _ in range(n_samples):
        start = np.random.uniform(-5, 5)
        if np.random.random() > 0.5:
            # Increasing
            seq = np.sort(np.random.uniform(start, start + 10, seq_len))
            noise = np.random.normal(0, 0.2, seq_len)
            X.append(seq + noise)
            y.append(1)
        else:
            # Decreasing
            seq = np.sort(np.random.uniform(start, start + 10, seq_len))[::-1]
            noise = np.random.normal(0, 0.2, seq_len)
            X.append(seq + noise)
            y.append(0)
    return torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32)

X_train, y_train = generate_sequences(2000)
X_test, y_test = generate_sequences(400)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Visualize a few samples
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(5):
    idx_inc = (y_train == 1).nonzero()[i].item()
    idx_dec = (y_train == 0).nonzero()[i].item()
    axes[0].plot(X_train[idx_inc].numpy(), alpha=0.7)
    axes[1].plot(X_train[idx_dec].numpy(), alpha=0.7)

axes[0].set_title('Increasing Sequences (Label=1)')
axes[1].set_title('Decreasing Sequences (Label=0)')
for ax in axes:
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, rnn_type='LSTM', input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.rnn_type = rnn_type

    def forward(self, x):
        if self.rnn_type == 'LSTM':
            output, (h_n, _) = self.rnn(x)
        else:
            output, h_n = self.rnn(x)
        return self.fc(h_n[-1]).squeeze(1)


def train_and_evaluate(rnn_type, X_train, y_train, X_test, y_test, epochs=30, lr=0.01):
    model = SimpleRNNClassifier(rnn_type=rnn_type)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    train_losses, test_accs = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        logits = model(X_train)
        loss = criterion(logits, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            preds = (torch.sigmoid(test_logits) >= 0.5).float()
            acc = (preds == y_test).float().mean().item()
            test_accs.append(acc)

    return train_losses, test_accs


results = {}
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    losses, accs = train_and_evaluate(rnn_type, X_train, y_train, X_test, y_test)
    results[rnn_type] = (losses, accs)
    print(f"{rnn_type:4s} — Final Test Accuracy: {accs[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (losses, accs) in results.items():
    axes[0].plot(losses, label=name, linewidth=2)
    axes[1].plot(accs, label=name, linewidth=2)

axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparing RNN, LSTM, and GRU on Sequence Classification', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Summary & Architecture Comparison

| Feature | Vanilla RNN | LSTM | GRU |
|---------|-------------|------|-----|
| Gates | None | 3 (Forget, Input, Output) | 2 (Reset, Update) |
| Cell state | No | Yes | No (merged into hidden) |
| Long-range deps | Poor | Good | Good |
| Parameters | Fewest | Most (~4x RNN) | Middle (~3x RNN) |
| Training speed | Fastest | Slowest | Middle |

### When to use what?
- **Vanilla RNN**: Short sequences, simple patterns
- **LSTM**: Long sequences, complex dependencies, when you need fine-grained control
- **GRU**: Similar to LSTM but faster; good default choice when dataset is smaller
- **Bidirectional**: When full context is available (classification, not real-time generation)